In [1]:
# 07_figures.ipynb
# Publication figures for the revision, in GRAYSCALE via seaborn, NO captions/titles,
# dpi 600, saved as BOTH .png and .pdf to ../results/figures. Run notebook 06 first.
#  - Per-classifier critical-difference (CD) diagrams (R8-1)
#  - Mean AUC with 95% CI error bars, per classifier (R8-4)
#  - AUC distribution across seeds (boxplot), per classifier (R8-4)
#  - Q-G-CTGAN (full) vs G-CTGAN (unfiltered) per-dataset delta AUC (R8-2)
import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

TABLES_DIR = "../results/tables"
FIG_DIR = "../results/figures"
os.makedirs(FIG_DIR, exist_ok=True)
CLASSIFIERS = ("RF", "LGBM", "MLP")
LABELS = {"None": "None", "SMOTE": "SMOTE", "ADASYN": "ADASYN", "G-SMOTE": "G-SMOTE",
          "KMeans_CTGAN": "K-means CTGAN", "CTGAN_MOS": "CTGAN-MOS", "ctdGAN": "ctdGAN",
          "CTAB-GAN+": "CTAB-GAN+", "G-CTGAN": "G-CTGAN", "Q-G-CTGAN": "Q-G-CTGAN"}

# grayscale, print-friendly seaborn theme
sns.set_theme(style="whitegrid", context="paper")
plt.rcParams.update({"font.family": "sans-serif", "font.size": 10,
                     "axes.edgecolor": "black", "axes.linewidth": 0.8,
                     "pdf.fonttype": 42, "ps.fonttype": 42})
GRAY = sns.color_palette("Greys", n_colors=8)

def save(fig, stem):
    fig.savefig(os.path.join(FIG_DIR, stem + ".png"), dpi=600, bbox_inches="tight")
    fig.savefig(os.path.join(FIG_DIR, stem + ".pdf"), bbox_inches="tight")
    plt.close(fig)
    print("saved", stem + ".png/.pdf")

agg = pd.read_csv(os.path.join(TABLES_DIR, "05_aggregated_mean_std_ci.csv"))
raw = pd.read_csv(os.path.join(TABLES_DIR, "05_multiseed_raw.csv"))

def nemenyi_cd(k, n):
    q = {2:1.960,3:2.343,4:2.569,5:2.728,6:2.850,7:2.949,8:3.031,9:3.102,10:3.164}.get(k, 3.164)
    return q * np.sqrt(k * (k + 1) / (6.0 * n))

# ---- (1) per-classifier CD diagrams (grayscale) ----
for clf in CLASSIFIERS:
    ar = pd.read_csv(os.path.join(TABLES_DIR, f"avg_ranks_{clf}.csv"))
    nem = pd.read_csv(os.path.join(TABLES_DIR, f"nemenyi_{clf}.csv"), index_col=0)
    disp2key = {v: k for k, v in LABELS.items()}
    methods = [disp2key.get(m, m) for m in ar["method"]]
    ranks = dict(zip(methods, ar["avg_rank"]))
    sm = sorted(methods, key=lambda m: ranks[m]); n = len(sm); rs = [ranks[m] for m in sm]
    fig, ax = plt.subplots(figsize=(7.2, 2.8))
    ax.set_xlim(min(rs) - 0.6, max(rs) + 0.6); ax.set_ylim(0, n + 2)
    ax.axhline(n + 0.5, color="black", lw=1)
    for i, (m, r) in enumerate(zip(sm, rs)):
        yy = n - i; left = i < n / 2
        ax.plot([r, r], [yy, n + 0.5], color="0.4", lw=0.8, ls=":")
        ax.plot(r, n + 0.5, "o", color="black", ms=4)
        ax.text(r - 0.05 if left else r + 0.05, yy, f"{LABELS.get(m, m)} ({r:.2f})",
                va="center", ha="right" if left else "left", fontsize=9)
    by = 0.3
    for i in range(n):
        for j in range(i + 1, n):
            if float(nem.loc[sm[i], sm[j]]) >= 0.05:
                ax.plot([rs[i], rs[j]], [by, by], color="black", lw=2.2, solid_capstyle="butt"); by += 0.33
    ax.set_yticks([]); ax.set_xlabel("Average rank")
    for s in ["top", "left", "right"]: ax.spines[s].set_visible(False)
    save(fig, f"fig_cd_{clf}")

# ---- (2) mean AUC with 95% CI error bars, per classifier (grayscale) ----
MORDER = ["None","SMOTE","ADASYN","G-SMOTE","KMeans_CTGAN","CTGAN_MOS","ctdGAN","CTAB-GAN+","G-CTGAN","Q-G-CTGAN"]
for clf in CLASSIFIERS:
    s = agg[agg["classifier"] == clf].copy()
    s = s[s["method"].isin(MORDER)]
    s["mean_over_ds"] = s.groupby("method")["AUC_mean"].transform("mean")
    order = [m for m in MORDER if m in set(s["method"])]
    g = s.groupby("method").agg(mean=("AUC_mean", "mean"),
                                 half=("AUC_ci95_half", "mean")).reindex(order)
    fig, ax = plt.subplots(figsize=(7.2, 3.2))
    xpos = np.arange(len(order))
    ax.errorbar(xpos, g["mean"].values, yerr=g["half"].values, fmt="o", color="black",
                ecolor="0.5", capsize=3, ms=5)
    ax.set_xticks(xpos); ax.set_xticklabels([LABELS.get(m, m) for m in order], rotation=40, ha="right")
    ax.set_ylabel("Mean AUC (95% CI)")
    sns.despine(ax=ax)
    save(fig, f"fig_auc_mean_ci_{clf}")

# ---- (3) AUC distribution across seeds, per classifier (boxplot, grayscale) ----
for clf in CLASSIFIERS:
    d = raw[(raw["classifier"] == clf) & (raw["method"].isin(MORDER))].copy()
    order = [m for m in MORDER if m in set(d["method"])]
    d["method"] = pd.Categorical(d["method"], categories=order, ordered=True)
    fig, ax = plt.subplots(figsize=(7.6, 3.4))
    sns.boxplot(data=d, x="method", y="AUC", order=order, color="0.8",
                linecolor="black", linewidth=0.9, fliersize=2, ax=ax)
    ax.set_xticks(range(len(order)))
    ax.set_xticklabels([LABELS.get(m, m) for m in order], rotation=40, ha="right")
    ax.set_xlabel(""); ax.set_ylabel("AUC (across seeds x datasets)")
    sns.despine(ax=ax)
    save(fig, f"fig_auc_box_{clf}")

# ---- (4) full vs unfiltered per-dataset delta AUC (R8-2, grayscale) ----
det_path = os.path.join(TABLES_DIR, "r8_2_per_dataset_detail.csv")
if os.path.exists(det_path):
    det = pd.read_csv(det_path)
    for clf in CLASSIFIERS:
        s = det[det["classifier"] == clf].sort_values("delta")
        if not len(s): continue
        fig, ax = plt.subplots(figsize=(7.2, 3.4))
        colors = ["0.3" if v >= 0 else "0.65" for v in s["delta"]]
        ax.barh(s["dataset"], s["delta"], color=colors, edgecolor="black", linewidth=0.6)
        ax.axvline(0, color="black", lw=0.8)
        ax.set_xlabel("AUC(Q-G-CTGAN) - AUC(G-CTGAN, unfiltered)")
        sns.despine(ax=ax)
        save(fig, f"fig_full_vs_unfiltered_{clf}")

print("DONE. Figures in", FIG_DIR)


saved fig_cd_RF.png/.pdf
saved fig_cd_LGBM.png/.pdf
saved fig_cd_MLP.png/.pdf
saved fig_auc_mean_ci_RF.png/.pdf
saved fig_auc_mean_ci_LGBM.png/.pdf
saved fig_auc_mean_ci_MLP.png/.pdf
saved fig_auc_box_RF.png/.pdf
saved fig_auc_box_LGBM.png/.pdf
saved fig_auc_box_MLP.png/.pdf
saved fig_full_vs_unfiltered_RF.png/.pdf
saved fig_full_vs_unfiltered_LGBM.png/.pdf
saved fig_full_vs_unfiltered_MLP.png/.pdf
DONE. Figures in ../results/figures
